# 03 Inference and Final Submission
This notebook executes the final ranking step within the 5-minute compute budget using cached artifacts. It runs completely offline.

In [34]:
%pip install pandas numpy sentence-transformers pyarrow fastparquet python-docx

Note: you may need to restart the kernel to use updated packages.


In [35]:
import os
import sys
import time
import json
import re
import numpy as np
import pandas as pd
import subprocess
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Resolve project root
base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

if base_dir not in sys.path:
    sys.path.insert(0, os.path.abspath(base_dir))

raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
outputs_dir = os.path.join(base_dir, "outputs", "submissions")
os.makedirs(outputs_dir, exist_ok=True)

# Import shared modules from src/
from src.data_loader import build_jd_text, load_candidates_for_ids
from src.embeddings import load_encoder_model
from src.scoring import compute_final_score, generate_reasoning

print(f"Project root: {os.path.abspath(base_dir)}")


Project root: /Users/koustav/ai-candidate-ranking


### 1. Load Processed Artifacts

In [36]:
print("Loading precomputed artifacts...")
start_time = time.time()

ids_path = os.path.join(processed_dir, "candidate_ids.npy")
candidate_ids = np.load(ids_path, allow_pickle=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
cand_embeddings = np.load(emb_path)

feat_path = os.path.join(processed_dir, "candidates_feather.parquet")
features_df = pd.read_parquet(feat_path)

print(f"Loaded {len(candidate_ids)} candidates.")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

Loading precomputed artifacts...
Loaded 100000 candidates.
Loading time: 0.14 seconds


### 2. Build and Encode JD Text

In [37]:
def build_jd_text(docx_path):
    try:
        doc = Document(docx_path)
        return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

# Load model avoiding network downloads by relying on local cache
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name)

print("Encoding JD text...")
jd_embedding = model.encode([jd_text]) if jd_text else np.array([])

Loading sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding JD text...


### 3. Compute Semantic Similarity

In [38]:
print("Computing cosine similarities...")
sim_start = time.time()

if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]
else:
    similarities = np.zeros(len(candidate_ids))

print(f"Similarity computation took: {time.time() - sim_start:.3f} seconds")

sim_df = pd.DataFrame({"candidate_id": candidate_ids, "semantic_similarity": similarities})
full_df = pd.merge(features_df, sim_df, on="candidate_id", how="inner")

# --- Normalize features that are out of [0,1] in the precomputed parquet ---
if full_df["github_fit_score"].max() > 1.0:
    full_df["github_fit_score"] = (full_df["github_fit_score"] / 100.0).clip(0, 1)
    print("  Normalized github_fit_score → [0,1]")
if full_df["availability_score"].max() > 1.0:
    amax = full_df["availability_score"].max()
    full_df["availability_score"] = (full_df["availability_score"] / amax).clip(0, 1)
    print(f"  Normalized availability_score → [0,1] (was 0–{amax:.1f})")
if full_df["ai_skill_depth_score"].max() > 1.0:
    dmax = full_df["ai_skill_depth_score"].max()
    full_df["ai_skill_depth_score"] = (full_df["ai_skill_depth_score"] / dmax).clip(0, 1)
    print(f"  Normalized ai_skill_depth_score → [0,1] (was 0–{dmax:.1f})")

print(f"\nMerged DataFrame: {full_df.shape[0]} rows, {full_df.shape[1]} cols")


Computing cosine similarities...
Similarity computation took: 0.116 seconds
  Normalized github_fit_score → [0,1]
  Normalized availability_score → [0,1] (was 0–11.1)
  Normalized ai_skill_depth_score → [0,1] (was 0–9.9)

Merged DataFrame: 100000 rows, 16 cols


### 4. Final Scoring & Reasoning Generation

In [39]:
# --- Scoring and Reasoning (imported from src/) ---
# compute_final_score and generate_reasoning are imported in Cell 2.
# They handle:
#   - Feature normalisation (all inputs clamped to [0,1])
#   - Weighted sum with total positive weights = 0.88
#   - Penalty subtraction (notice period α=0.15, honeypot β=0.30)
#   - JD-aware, evidence-backed reasoning with no duplicate phrasing

TOP_K = 100  # Per spec: exactly 100 candidates

# Score ALL candidates on full_df (not ranked_df)
max_ml = full_df["ml_years_estimate"].max()
if max_ml == 0: max_ml = 1.0

full_df["final_score"] = full_df.apply(
    lambda row: compute_final_score(row, max_ml_years=max_ml), axis=1
)

score_min = full_df["final_score"].min()
score_max = full_df["final_score"].max()
score_uniq = full_df["final_score"].nunique()
print(f"Full score range: {score_min:.4f} to {score_max:.4f}")
print(f"Unique scores: {score_uniq}/{len(full_df)}")

# Filter honeypots, then select top 100
filtered_df = full_df[full_df["honeypot_risk_score"] < 0.6].copy()
honeypot_count = len(full_df) - len(filtered_df)
print(f"Filtered out {honeypot_count} honeypot candidates (risk >= 0.6)")

# Sort by score DESC, tiebreak by candidate_id ASC
ranked_df = filtered_df.sort_values(
    by=["final_score", "candidate_id"],
    ascending=[False, True],
).head(TOP_K).copy()

print(f"Selected top {len(ranked_df)} from {len(filtered_df)} remaining")
top_min = ranked_df["final_score"].min()
top_max = ranked_df["final_score"].max()
print(f"Top-{TOP_K} score range: {top_min:.4f} to {top_max:.4f}")

# --- Load raw candidate details for top-100 reasoning ---
top_ids = set(ranked_df["candidate_id"].values)
cand_details = load_candidates_for_ids(os.path.join(raw_dir, "candidates.jsonl"), top_ids)
print(f"Loaded raw details for {len(cand_details)}/{len(top_ids)} candidates")

def safe_reasoning(row):
    cid = row["candidate_id"]
    return generate_reasoning(row, cand_raw=cand_details.get(cid))

ranked_df["reasoning"] = ranked_df.apply(safe_reasoning, axis=1)

uniq_reasons = ranked_df["reasoning"].nunique()
print(f"Unique reasonings: {uniq_reasons}/{len(ranked_df)}")
print()
print("=== 5 Random Reasoning Samples ===")
for _, r in ranked_df.sample(min(5, len(ranked_df)), random_state=42).iterrows():
    cid = r["candidate_id"]
    sc = r.get("final_score", 0)
    print(f"  {cid} | score={sc:.3f}")
    print(f"    {r['reasoning']}")
    print()


Full score range: 0.1745 to 0.6491
Unique scores: 100000/100000
Filtered out 0 honeypot candidates (risk >= 0.6)
Selected top 100 from 100000 remaining
Top-100 score range: 0.5429 to 0.6491
Loaded raw details for 100/100 candidates
Unique reasonings: 100/100

=== 5 Random Reasoning Samples ===
  CAND_0052335 | score=0.547
    Applied ML Engineer with 6 years of experience, 2.7 of which in ML-related roles. Relevant to the JD through search/retrieval (Semantic Search) and embeddings/vectors (Sentence Transformers). Platform: 79% recruiter response rate, open to work.

  CAND_0033911 | score=0.558
    Data Scientist with 7 years of experience, 5.7 of which in ML-related roles. Relevant to the JD through embeddings/vectors (FAISS, Sentence Transformers) and MLOps (MLOps). Platform: 76% recruiter response rate. Note: 120-day notice period.

  CAND_0033445 | score=0.552
    ML Engineer with 7 years of experience, 3.3 of which in ML-related roles. Relevant to the JD through search/retrieval 

### 5. Format Submission

In [40]:
ranked_df["score"] = ranked_df["final_score"].round(4)
ranked_df["rank"] = range(1, len(ranked_df) + 1)

# Enforce monotonically non-increasing scores
prev_score = ranked_df.iloc[0]["score"]
adjusted_scores = []
for s in ranked_df["score"]:
    if s > prev_score:
        adjusted_scores.append(prev_score)
    else:
        adjusted_scores.append(s)
        prev_score = s
ranked_df["score"] = adjusted_scores

# --- Score sanity checks ---
n_unique = len(set(adjusted_scores))
assert n_unique > 1, (
    f"SCORING BUG: All {len(adjusted_scores)} scores are identical. "
    "The scoring formula is saturating."
)
print(f"Score diversity: {n_unique} unique values across {len(adjusted_scores)} candidates")
print(f"Score range: {min(adjusted_scores):.4f} to {max(adjusted_scores):.4f}")

# Build submission -- exactly 4 columns per submission_spec.docx
submission_df = ranked_df[["candidate_id", "rank", "score", "reasoning"]].copy()

# ===== Spec guardrails (exactly 100 rows per spec) =====
n_expected = 100
assert len(submission_df) == n_expected, f"Expected {n_expected} rows, got {len(submission_df)}"

ranks = submission_df["rank"].tolist()
assert sorted(set(ranks)) == list(range(1, n_expected + 1)), "Ranks must be 1..100 with no gaps or duplicates"

ids = submission_df["candidate_id"].tolist()
assert len(set(ids)) == len(ids), "Duplicate candidate_id values found in submission"

# Check all IDs exist in the original candidates file
cand_ids_set = set()
cands_path_check = os.path.join(raw_dir, "candidates.jsonl")
with open(cands_path_check, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cand = json.loads(line)
        cid = cand.get("candidate_id")
        if cid:
            cand_ids_set.add(cid)

missing = [cid for cid in ids if cid not in cand_ids_set]
assert not missing, f"{len(missing)} candidate_ids not found in candidates.jsonl: {missing[:5]}"

# Score monotonicity
scores = submission_df["score"].tolist()
mono = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))
assert mono, "Scores must be monotonically non-increasing with rank"

assert len(set(scores)) > 1, "All scores are identical -- scoring bug"

# No NaN or empty values
assert submission_df.isna().sum().sum() == 0, "NaN values found"
assert (submission_df["reasoning"].str.strip() != "").all(), "Empty reasoning found"

print()
print("=" * 50)
print("  All spec guardrails passed!")
print("=" * 50)
print(f"Columns: {list(submission_df.columns)}")
print(f"Rows: {len(submission_df)}")

sub_path = os.path.join(outputs_dir, "ranked_candidates.csv")
submission_df.to_csv(sub_path, index=False)
print(f"Saved final submission to {sub_path}")


Score diversity: 87 unique values across 100 candidates
Score range: 0.5429 to 0.6491

  All spec guardrails passed!
Columns: ['candidate_id', 'rank', 'score', 'reasoning']
Rows: 100
Saved final submission to ../outputs/submissions/ranked_candidates.csv


### 6. Validation and Inspection

In [41]:
# --- Validation using the OFFICIAL validator from data/raw/ ---
import sys

val_script = os.path.join(raw_dir, "validate_submission.py")
assert os.path.exists(val_script), f"Official validator not found at {val_script}"

meta_path = os.path.join(base_dir, "submission_metadata.yaml")
if not os.path.exists(meta_path):
    meta_path = os.path.join(raw_dir, "submission_metadata_template.yaml")
assert os.path.exists(meta_path), f"Metadata YAML not found at {meta_path}"

print(f"Validator : {val_script}")
print(f"Metadata  : {meta_path}")
print(f"Submission: {sub_path}")
print(f"Columns   : {list(submission_df.columns)}")

print()
print("=== Running Official Validator ===")
try:
    result = subprocess.run(
        [sys.executable, val_script, sub_path, meta_path],
        capture_output=True, text=True
    )
    stdout = result.stdout.strip()
    stderr = result.stderr.strip()
    print(stdout)
    if stderr:
        print(f"STDERR: {stderr}")

    if result.returncode == 0 and "successful" in stdout.lower():
        print()
        print("Validation PASSED")
        validated_path = os.path.join(outputs_dir, "ranked_candidates_validated.csv")
        submission_df.to_csv(validated_path, index=False)
        print(f"Saved validated copy to {validated_path}")
    else:
        err_msg = stderr if stderr else stdout
        print()
        print(f"Validation FAILED: {err_msg}")
        print("ranked_candidates_validated.csv was NOT overwritten.")

except Exception as e:
    print()
    print(f"Validation FAILED: {e}")
    print("ranked_candidates_validated.csv was NOT overwritten.")

# --- Inspection ---
print()
print("=== Submission Summary ===")
print(f"Rows: {len(submission_df)} | Unique IDs: {submission_df.candidate_id.nunique()}")
print(f"Columns: {list(submission_df.columns)}")
s_min = submission_df.score.min()
s_max = submission_df.score.max()
print(f"Score range: {s_min:.4f} to {s_max:.4f}")
p50 = submission_df.score.quantile(0.5)
p90 = submission_df.score.quantile(0.9)
print(f"Score percentiles: p50={p50:.4f}, p90={p90:.4f}")
r_min = submission_df["rank"].min()
r_max = submission_df["rank"].max()
print(f"Ranks: {r_min} to {r_max}")
mono = all(submission_df.score.iloc[i] >= submission_df.score.iloc[i+1] for i in range(len(submission_df)-1))
mono_label = "YES" if mono else "NO"
print(f"Scores monotonically non-increasing: {mono_label}")
uniq_r = submission_df.reasoning.nunique()
print(f"Unique reasonings: {uniq_r}/{len(submission_df)}")

print()
print("=== Top 10 rows ===")
display(submission_df.head(10))

print()
print("=== Bottom 5 rows ===")
display(submission_df.tail(5))

print()
print("=== 5 Random Sample rows ===")
display(submission_df.sample(5, random_state=42))


Validator : ../data/raw/validate_submission.py
Metadata  : ../submission_metadata.yaml
Submission: ../outputs/submissions/ranked_candidates.csv
Columns   : ['candidate_id', 'rank', 'score', 'reasoning']

=== Running Official Validator ===
Validating ../outputs/submissions/ranked_candidates.csv...
Checking metadata...
Validation successful!

Validation PASSED
Saved validated copy to ../outputs/submissions/ranked_candidates_validated.csv

=== Submission Summary ===
Rows: 100 | Unique IDs: 100
Columns: ['candidate_id', 'rank', 'score', 'reasoning']
Score range: 0.5429 to 0.6491
Score percentiles: p50=0.5595, p90=0.5935
Ranks: 1 to 100
Scores monotonically non-increasing: YES
Unique reasonings: 100/100

=== Top 10 rows ===


,candidate_id,rank,score,reasoning
81845,CAND_0081846,1,0.6491,"Lead AI Engineer with 7 years of experience, 4..."
80765,CAND_0080766,2,0.6278,Staff Machine Learning Engineer with 9 years o...
98845,CAND_0098846,3,0.6225,AI Engineer with 8 years of experience in a pr...
55904,CAND_0055905,4,0.6150,Senior Machine Learning Engineer with 8 years ...
93192,CAND_0093193,5,0.6149,Senior Machine Learning Engineer with 8 years ...
83306,CAND_0083307,6,0.6132,Search Engineer with 8 years of experience in ...
7008,CAND_0007009,7,0.6127,Recommendation Systems Engineer with 8 years o...
87629,CAND_0087630,8,0.6081,AI Engineer with 7 years of experience in a pr...
18498,CAND_0018499,9,0.6053,Senior Machine Learning Engineer with 7 years ...
6566,CAND_0006567,10,0.5976,"Senior AI Engineer with 8 years of experience,..."



=== Bottom 5 rows ===


,candidate_id,rank,score,reasoning
421,CAND_0000422,96,0.5439,AI Research Engineer with 6 years of experienc...
18292,CAND_0018293,97,0.5435,"Data Scientist with 6 years of experience, 4.2..."
4401,CAND_0004402,98,0.5431,AI Research Engineer with 6 years of experienc...
55991,CAND_0055992,99,0.5429,AI Engineer with 17 years of experience in a p...
24619,CAND_0024620,100,0.5429,"AI Engineer with 6 years of experience, 2.7 of..."



=== 5 Random Sample rows ===


,candidate_id,rank,score,reasoning
52334,CAND_0052335,84,0.5469,Applied ML Engineer with 6 years of experience...
33910,CAND_0033911,54,0.5582,"Data Scientist with 7 years of experience, 5.7..."
33444,CAND_0033445,71,0.5523,"ML Engineer with 7 years of experience, 3.3 of..."
79386,CAND_0079387,46,0.5632,"AI Engineer with 7 years of experience, 3.0 of..."
30030,CAND_0030031,45,0.5636,"AI Engineer with 6 years of experience, 4.4 of..."
